# Examen en Vivo (Google Colab) — Visión Artificial Clásica: Baraja Española (Reto 07)

Cuaderno preparado para ejecutar las pruebas del profesor en **Google Colab** sin depender de ventanas locales (`tkinter` / `cv2.imshow`).

**Cómo usarlo el día del examen en 3 pasos:**
1. Ejecuta la **Celda 1** una vez al empezar (clona tu repositorio de GitHub y carga el motor clásico).
2. Ejecuta la **Celda 2** para verificar que todo funciona sobre `fotos/IMG_7890.jpeg`.
3. Cada vez que el profesor coloque cartas en el tapete y hagas una foto con el móvil, ejecuta la **Celda 3**, sube la foto y verás al instante el resultado, la ruta del árbol de decisión y el diagnóstico homográfico.
4. *(Solo si la luz del aula difiere mucho de tu caja)* Ejecuta la **Celda 4** para revisar la máscara binaria del tapete y ajustar en segundos `UMBRAL_V_SOMBRA_FONDO` o `VERDE_BAJO`/`VERDE_ALTO`.

## Celda 1 — Clonar / actualizar repositorio e inicializar utilidades de visualización

In [ ]:
import os
import subprocess
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/xImRaul/vision-artificial-baraja-espanola.git'
REPO_DIR = '/content/vision-artificial-baraja-espanola'

if 'google.colab' in sys.modules or os.path.isdir('/content'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

import config as cfg
from pipeline_baraja import (
    leer_imagen,
    guardar_imagen,
    procesar_escena,
    dibujar_resultados,
    crear_mosaico_diagnostico,
    segmentar_tapete,
)

ULTIMA_IMAGEN_ANALIZADA = 'fotos/IMG_7890.jpeg'

def analizar_foto_colab(ruta_img, mostrar_diagnostico=True, guardar_salida=True):
    """Ejecuta el pipeline clásico completo sobre una foto y muestra todo en Colab."""
    global ULTIMA_IMAGEN_ANALIZADA
    img = leer_imagen(ruta_img)
    if img is None:
        raise FileNotFoundError(f'No se pudo leer la imagen: {ruta_img}')
    ULTIMA_IMAGEN_ANALIZADA = ruta_img

    h_img, w_img = img.shape[:2]
    area_img = float(h_img * w_img)
    res = procesar_escena(img)
    interp = res['interpretacion']

    print('=' * 78)
    print(f'ESCENA ANALIZADA : {ruta_img} ({w_img}x{h_img} px)')
    print(f'COBERTURA TAPETE : {res["porcentaje_tapete"]:.1f}%')
    if res['cartas']:
        escala = float(np.mean([c['px_por_mm'] for c in res['cartas']]))
        print(f'ESCALA ESPACIAL  : {escala:.2f} px/mm (FOV est: {w_img/escala/10:.1f} x {h_img/escala/10:.1f} cm)')
    print('-' * 78)
    print('NIVEL 1 (CONTORNOS Y GEOMETRIA) + NIVEL 2 (ARBOL DE DECISION):')
    if not res['cartas']:
        print('  (0 cartas detectadas sobre el tapete)')
    for c in res['cartas']:
        f = c['caracteristicas']
        frac_pct = 100.0 * c['area_carta_px2'] / area_img
        print(
            f"  #{c['id']:02d} {c['nombre']:17s} | Area={c['area_carta_px2']:.0f}px ({frac_pct:.2f}%) "
            f"AR={c['aspecto_carta']:.2f} Rect={c['rectangularidad']:.3f} Sol={c['solidez_externa']:.3f} | "
            f"Pos=({c['posicion_cm'][0]:.1f},{c['posicion_cm'][1]:.1f})cm Giro={c['angulo_360_deg']:.1f}deg"
        )
        print(f"      Ruta Arbol: {' ; '.join(c['ruta_arbol'])}")

    print('-' * 78)
    print('NIVEL 3 (INTERPRETACION SEMANTICA DE LA MANO):')
    cp = interp['conteo_palos']
    print(f"  Total: {interp['total_cartas']} | Oros:{cp['Oros']} Copas:{cp['Copas']} Espadas:{cp['Espadas']} Bastos:{cp['Bastos']}")
    print(f"  Valores: {interp['conteo_valores']} | Suma puntos: {interp['suma_puntos']}")
    print(f"  Jugada: {interp['jugada_mano']} | 7 y Media: {interp['estado_7ymedia']}")
    escobas = ', '.join(interp['escobas_15']) if interp['escobas_15'] else 'Ninguna'
    print(f'  Escobas (15): {escobas}')
    print('=' * 78)

    anotada = dibujar_resultados(img, res)
    if guardar_salida:
        base, ext = os.path.splitext(ruta_img)
        ruta_out = f'{base}_resultado{ext or ".jpg"}'
        guardar_imagen(ruta_out, anotada)
        print(f'Imagen resultado guardada en: {ruta_out}')

    plt.figure(figsize=(13, 9), dpi=110)
    plt.imshow(cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB))
    plt.title(f"{os.path.basename(ruta_img)} — {interp['total_cartas']} cartas | Suma: {interp['suma_puntos']} pts ({interp['jugada_mano']})", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    if mostrar_diagnostico and res['cartas']:
        diag = crear_mosaico_diagnostico(res)
        plt.figure(figsize=(14, 7), dpi=110)
        plt.imshow(cv2.cvtColor(diag, cv2.COLOR_BGR2RGB))
        plt.title('Mosaico Diagnostico: Homografias 300x460 px y Mascaras Internas', fontsize=12, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    return res

print('Entorno listo para analizar fotos en Google Colab.')

## Celda 2 — Prueba rápida de verificación con `fotos/IMG_7890.jpeg`

In [ ]:
_ = analizar_foto_colab('fotos/IMG_7890.jpeg', mostrar_diagnostico=True, guardar_salida=False)

## Celda 3 — EXAMEN EN VIVO: Subir nueva fotografía del profesor
Ejecuta esta celda cada vez que hagas una foto nueva cambiando cartas de posición o rotación.

In [ ]:
# Inicializacion ultrarrapida (< 0.5 s): descarga solo los 3 archivos de codigo (35 KB) si no estan cargados
if 'analizar_foto_colab' not in globals():
    import os, sys, urllib.request, cv2, matplotlib.pyplot as plt
    RAW_BASE = 'https://raw.githubusercontent.com/xImRaul/vision-artificial-baraja-espanola/main/'
    for fname in ['config.py', 'pipeline_baraja.py', 'calibracion_tapete.json']:
        if not os.path.exists(fname):
            urllib.request.urlretrieve(RAW_BASE + fname, fname)
    sys.path.insert(0, os.getcwd())
    from pipeline_baraja import procesar_escena, dibujar_resultados, crear_mosaico_diagnostico
    def analizar_foto_colab(ruta_imagen, mostrar_diagnostico=True, guardar_salida=True):
        img_bgr = cv2.imread(ruta_imagen)
        if img_bgr is None:
            print(f'No se pudo abrir la imagen: {ruta_imagen}')
            return
        res = procesar_escena(img_bgr)
        vis = dibujar_resultados(img_bgr, res)
        interp = res['interpretacion']
        print('=' * 70)
        print(f'ARCHIVO ANALIZADO : {ruta_imagen}')
        print(f'CARTAS DETECTADAS : {interp["total_cartas"]}')
        for idx, c in enumerate(res['cartas'], 1):
            print(f'  [{idx}] {c["nombre"]} ({c["codigo"]}) | Giro: {c["angulo"]:.0f} deg')
        print(f'SIETE Y MEDIA     : {interp["estado_7ymedia"]}')
        print(f'ESCOBAS (SUMA 15) : {interp["escobas_15"]}')
        print(f'JUGADA DE MANO    : {interp["jugada_mano"]}')
        print('=' * 70)
        plt.figure(figsize=(14, 10))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f'{ruta_imagen} -> {interp["total_cartas"]} cartas | {interp["jugada_mano"]}')
        plt.show()
        if mostrar_diagnostico:
            diag = crear_mosaico_diagnostico(img_bgr, res)
            plt.figure(figsize=(14, 10))
            plt.imshow(cv2.cvtColor(diag, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.title('Mosaico de Diagnostico (4 Etapas del Pipeline)')
            plt.show()

try:
    from google.colab import files
    print('>>> BAJA UN POCO HACIA ABAJO Y PULSA EL BOTON "Choose Files / Elegir archivos" <<<', flush=True)
    subidas = files.upload()
    for nombre_archivo in subidas.keys():
        analizar_foto_colab(nombre_archivo, mostrar_diagnostico=True, guardar_salida=True)
except ImportError:
    print('Esta celda usa google.colab.files.upload(). Fuera de Colab, llama directamente a:')
    print("analizar_foto_colab('ruta/a/tu_foto.jpg')")


## Celda 4 — Diagnóstico de máscara HSV y ajuste rápido ante cambios de luz en el aula
Si al hacer una foto en el aula **sin caja / sin flash** alguna esquina oscura del tapete entra o sale de la máscara, ejecuta esta celda para verlo en blanco y negro y ajustar los 3 parámetros clave.

In [ ]:
# Ajusta aquí solo si cambia mucho la luz del aula respecto a tu caja con flash:
cfg.VERDE_BAJO = np.array([35, 22, 5], dtype=np.uint8)
cfg.VERDE_ALTO = np.array([98, 255, 255], dtype=np.uint8)
cfg.UMBRAL_V_SOMBRA_FONDO = 120  # Ejemplo: baja a 70 si el aula es mas oscura sin flash

img_diag = leer_imagen(ULTIMA_IMAGEN_ANALIZADA)
mask_obj, hsv_diag, cob = segmentar_tapete(img_diag)
fondo_mask = (mask_obj == 0)
if np.any(fondo_mask):
    pix_f = hsv_diag[fondo_mask]
    print(f'Cobertura tapete reconocido: {cob:.1f}%')
    print('Percentiles P1..P99 del tapete -> H:', np.percentile(pix_f[:,0], [1,99]),
          'S:', np.percentile(pix_f[:,1], [1,99]),
          'V:', np.percentile(pix_f[:,2], [1,99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=110)
axes[0].imshow(cv2.cvtColor(img_diag, cv2.COLOR_BGR2RGB))
axes[0].set_title('Imagen Original')
axes[0].axis('off')
axes[1].imshow(mask_obj, cmap='gray')
axes[1].set_title(f'Mascara Binaria Objetos (Tapete={cob:.1f}% | V_sombra<{cfg.UMBRAL_V_SOMBRA_FONDO})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Re-ejecutar análisis con los nuevos parámetros si la imagen contiene cartas:
_ = analizar_foto_colab(ULTIMA_IMAGEN_ANALIZADA, mostrar_diagnostico=False, guardar_salida=False)

## Celda 5 — Evaluación cuantitativa completa del banco de pruebas (47 cartas reales)

In [ ]:
subprocess.run([sys.executable, '3_evaluacion_cuantitativa.py', '--gt', 'fotos/ground_truth.json'], check=True)

fig_paths = [
    ('memoria_figuras/separabilidad_arbol_decision.png', 'Separabilidad Experimental y Umbrales del Arbol'),
    ('memoria_figuras/matriz_confusion_palos.png', 'Matriz de Confusion — Palos'),
    ('memoria_figuras/matriz_confusion_valores.png', 'Matriz de Confusion — Valores (1-7)'),
]
for ruta_f, tit in fig_paths:
    if os.path.exists(ruta_f):
        im_f = plt.imread(ruta_f)
        plt.figure(figsize=(11 if 'separabilidad' in ruta_f else 6, 4.8), dpi=120)
        plt.imshow(im_f)
        plt.title(tit)
        plt.axis('off')
        plt.tight_layout()
        plt.show()